# 03 — Results analysis (Chapter 5)

Builds every Chapter 5 table and figure from the committed pipeline outputs. All computation lives in
`src/results_analysis.py`; this notebook only calls it, shows the results and saves them to
`dissertation/tables/` and `dissertation/figures/` (interactive `.html`, plus `.png` when plotly's exporter is available).

**Prerequisites:** `make prepare`, `make sweep`, `make profile` — i.e. `data/processed/`, `experiments/results_raw.csv`
and `experiments/latency_results.csv` exist.

**Two stages, kept strictly apart (rule 3):**
1. **Cross-validation** on the training split ranks all 41 (technique, classifier) pairs → Table 2, Figure 8, Table 4.
2. **Final evaluation**: the top 5 pairs by CV AUC-PR plus the best untreated baseline are refitted on the full
   training split. Thresholds and operating points are chosen on **validation** and applied unchanged to **test**.
   Test is used once, here, and nothing is selected by looking at it.

In [4]:
import logging
import sys

sys.path.insert(0, "..")
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s")

import pandas as pd
from IPython.display import display

from src import config
from src import results_analysis as ra
from src.data_loader import FraudDataset

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 60)

## 5.2 Technique comparison — Table 2

Mean over the 5 CV folds, sorted by AUC-PR (primary metric, rule 2). `f1`, `mcc` and `precision_at_100` use a
threshold chosen on the same fold, so they are optimistic; AUC-PR is threshold-free and unaffected. Accuracy is
deliberately absent.

In [5]:
results = ra.load_results()
table2 = ra.cv_summary(results)
ra.save_table(table2, "table2_cv_results")
display(table2)

,rank,technique,classifier,type,auc_pr,auc_pr_std,auc_roc,precision_at_100,f1,mcc,fit_seconds,n_train
0,1,random_oversampling,xgboost,data-level,0.8611,0.0253,0.9751,0.5700,0.8774,0.8794,0.8225,308211.2000
1,2,xgb_scale_pos_weight,xgboost,algorithm-level,0.8592,0.0266,0.9740,0.5680,0.8822,0.8850,0.5538,154370.4000
2,3,xgb_focal_loss,xgboost,algorithm-level,0.8560,0.0359,0.9782,0.5740,0.8784,0.8798,1.6241,154370.4000
3,4,smote_enn,xgboost,data-level,0.8518,0.0218,0.9749,0.5600,0.8803,0.8848,1.9390,308211.2000
4,5,smote,xgboost,data-level,0.8511,0.0256,0.9719,0.5700,0.8779,0.8818,0.8839,308211.2000
5,6,smote_tomek,xgboost,data-level,0.8511,0.0256,0.9719,0.5700,0.8779,0.8818,1.1273,308211.2000
6,7,random_oversampling,random_forest,data-level,0.8503,0.0289,0.9454,0.5760,0.8710,0.8735,11.2719,308211.2000
7,8,smote_enn,random_forest,data-level,0.8498,0.0201,0.9724,0.5700,0.8670,0.8693,54.4189,308211.2000
8,9,soa_s,xgboost,data-level,0.8485,0.0263,0.9748,0.5580,0.8706,0.8762,1.2319,308211.2000
9,10,class_weight_balanced,random_forest,algorithm-level,0.8480,0.0296,0.9538,0.5740,0.8706,0.8725,7.6656,154370.4000


## 5.4 Latency — Figure 8 and Table 3

**Figure 8** is the novel-contribution plot: CV AUC-PR against single-record p99 latency, with the 100 ms budget.
Upper-left = accurate *and* deployable. It plots AUC-PR, not accuracy — the "accuracy vs latency" label in the plan
would contradict rule 2.

Colour = kind of treatment, marker = classifier, error bars = CV std; hover for exact pairs. Things to look for
when writing §5.4: whether data-level techniques sit at the same latency as their untreated classifier (the
hypothesis), how far right the ensembles sit, and whether anything crosses the budget.

In [6]:
latency = ra.load_latency()
table3 = ra.latency_table(latency, results)
ra.save_table(table3, "table3_latency")

fig8 = ra.fig_auc_pr_vs_latency(table3)
ra.save_figure(fig8, "figure8_auc_pr_vs_latency")
fig8.show()
display(table3)

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp2oqhmau1.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpsb1pzqvh.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpsb1pzqvh
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp2oqhmau1/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 58A8
INFO    Reloading tab 58A8 before return.
INFO    Putting tab 58A8 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

,technique,classifier,type,p50_ms,p95_ms,p99_ms,mean_ms,model_size_mb,cpu_utilisation,auc_pr,auc_pr_std,within_budget
0,smote,decision_tree,data-level,0.0487,0.0525,0.0589,0.0496,0.0903,1.0739,0.3042,0.0587,True
1,none,decision_tree,untreated,0.0469,0.0508,0.0630,0.0478,0.0177,1.0060,0.5750,0.0336,True
2,random_oversampling,logistic_regression,data-level,0.0542,0.0565,0.0645,0.0547,0.0008,1.0674,0.7269,0.0444,True
3,smote,logistic_regression,data-level,0.0540,0.0566,0.0650,0.0547,0.0008,1.0615,0.7275,0.0528,True
4,none,logistic_regression,untreated,0.0542,0.0579,0.0700,0.0552,0.0008,1.0618,0.7580,0.0276,True
5,random_undersampling,logistic_regression,data-level,0.0546,0.0585,0.0733,0.0558,0.0008,1.0786,0.4955,0.1568,True
6,smote_enn,logistic_regression,data-level,0.0539,0.0583,0.0782,0.0553,0.0008,1.0655,0.7275,0.0528,True
7,smote_tomek,decision_tree,data-level,0.0486,0.0532,0.0787,0.0501,0.0903,1.0719,0.3042,0.0587,True
8,random_oversampling,decision_tree,data-level,0.0471,0.0550,0.0835,0.0494,0.0228,1.0358,0.5431,0.0605,True
9,soa_s,decision_tree,data-level,0.0490,0.0625,0.0852,0.0517,0.0880,1.0503,0.3252,0.0420,True


Per-record cost (ms) by batch size: how much batching amortises the per-call overhead. Relevant to the API's
batch endpoint and to bulk re-scoring, not to the real-time budget.

In [7]:
table3b = ra.throughput_table(latency)
ra.save_table(table3b.reset_index(), "table3b_per_record_ms_by_batch")
display(table3b)

,batch_size,1,100,1000,10000
technique,classifier,,,,
none,decision_tree,0.0478,0.0006,0.0001,0.0001
random_oversampling,decision_tree,0.0494,0.0006,0.0001,0.0000
smote,decision_tree,0.0496,0.0006,0.0001,0.0001
smote_tomek,decision_tree,0.0501,0.0006,0.0001,0.0001
adasyn,decision_tree,0.0502,0.0006,0.0001,0.0001
smote_enn,decision_tree,0.0504,0.0006,0.0001,0.0001
random_undersampling,decision_tree,0.0511,0.0006,0.0001,0.0000
soa_s,decision_tree,0.0517,0.0006,0.0001,0.0001
class_weight_balanced,decision_tree,0.0523,0.0006,0.0001,0.0000


## 5.3 Statistical significance — Friedman–Nemenyi (Table 4)

Two comparisons:
- **(a) Data-level techniques**, blocked by classifier × fold (4 × 5 = 20 blocks): does a resampling technique help
  whatever the classifier? `differs_from_reference` flags a significant difference from no resampling.
- **(b) Within each base classifier**, every technique available for it (including its algorithm-level variants)
  over 5 folds, against that classifier's untreated model.

Caveats to state in §5.3: the blocks are CV folds, which share training data, so the tests are somewhat liberal
(Dietterich 1998); and 5 folds give little power, so "not significant" is weak evidence of equivalence.
Algorithm-level techniques outside the four base classifiers (RUSBoost, EasyEnsemble, LightGBM) have no untreated
counterpart, so they appear only in Table 2 and Figure 8.

In [8]:
data_level = ra.rank_comparison(
    ra.data_level_scores(results),
    "Data-level techniques (classifier × fold)",
    reference="none",
)
ra.save_table(data_level.table, "table4a_ranks_data_level")
display(data_level.table)
fig = ra.fig_mean_ranks(data_level)
ra.save_figure(fig, "figure_ranks_data_level")
fig.show()

,treatment,mean_rank,mean_auc_pr,worse_than_best,differs_from_reference
0,random_oversampling,2.5000,0.7454,False,False
1,none,3.7000,0.7406,False,False
2,smote_enn,3.7500,0.6833,False,False
3,smote,4.1500,0.6821,False,False
4,smote_tomek,4.1500,0.6821,False,False
5,soa_s,4.4000,0.6848,False,False
6,adasyn,5.3500,0.6694,True,False
7,random_undersampling,8.0000,0.4901,True,True


INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp64ajogjt.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpwb83lc6p.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpwb83lc6p
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp64ajogjt/index.html
INFO    Getting tab from queue (has 1)
INFO    Got AE94
INFO    Reloading tab AE94 before return.
INFO    Putting tab AE94 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

In [9]:
families = ra.family_rankings(results)
table4b = pd.concat(
    [r.table.assign(classifier=clf, friedman_p=r.p_value, cd=r.critical_difference)
     for clf, r in families.items()],
    ignore_index=True,
)
ra.save_table(table4b, "table4b_ranks_by_classifier")
display(table4b)
for clf, r in families.items():
    fig = ra.fig_mean_ranks(r)
    ra.save_figure(fig, f"figure_ranks_{clf}")
    fig.show()

,treatment,mean_rank,mean_auc_pr,worse_than_best,differs_from_reference,classifier,friedman_p,cd
0,none,2.4000,0.7580,False,False,logistic_regression,0.0016,5.3728
1,smote,3.4000,0.7275,False,False,logistic_regression,0.0016,5.3728
2,smote_enn,3.4000,0.7275,False,False,logistic_regression,0.0016,5.3728
3,smote_tomek,3.4000,0.7275,False,False,logistic_regression,0.0016,5.3728
4,class_weight_balanced,4.6000,0.7279,False,False,logistic_regression,0.0016,5.3728
5,random_oversampling,5.6000,0.7269,False,False,logistic_regression,0.0016,5.3728
6,soa_s,6.4000,0.7216,False,False,logistic_regression,0.0016,5.3728
7,adasyn,6.8000,0.7082,False,False,logistic_regression,0.0016,5.3728
8,random_undersampling,9.0000,0.4955,True,True,logistic_regression,0.0016,5.3728
9,none,1.6000,0.5750,False,False,decision_tree,0.0000,5.3728


INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpayugrjbt.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp9c2g2qk7.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp9c2g2qk7
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpayugrjbt/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 79A8
INFO    Reloading tab 79A8 before return.
INFO    Putting tab 79A8 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpmu8e_r8u.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpifo3hz4_.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpifo3hz4_
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpmu8e_r8u/index.html
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Getting tab from queue (has 1)
INFO    Got 69F3
INFO    Reloading tab 69F3 before return.
INFO    Putting tab 69F3 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browse

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpz9xrjfgx.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpcjk407dq.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpcjk407dq
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpz9xrjfgx/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 8127
INFO    Reloading tab 8127 before return.
INFO    Putting tab 8127 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp0ldllyoy.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmptzy_6vzf.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmptzy_6vzf
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp0ldllyoy/index.html
INFO    Getting tab from queue (has 1)
INFO    Got F6C7
INFO    Reloading tab F6C7 before return.
INFO    Putting tab F6C7 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worke

## 5.5 Final models on the test split (Table 5)

The top 5 pairs by CV AUC-PR, plus the best untreated baseline as the reference to beat. Each is refitted on the
whole training split (resampling the training data only), its max-F1 threshold is chosen on validation, and it is
scored once on test.

`auc_pr_lo/hi` is a stratified bootstrap 95% CI. `delta_vs_selected` is the difference from the CV-selected best
model on the **same** bootstrap rows, with its own CI; `significant` means that interval excludes zero. Overlapping
individual CIs do *not* by themselves mean "no difference" — the paired interval is the right test.

Refitting takes a few minutes if a random forest is among them.

In [10]:
splits, manifest = FraudDataset.load_splits(config.DATA_PROCESSED)

pairs = ra.top_pairs(results)
reference_pair = ra.best_untreated(results)
if reference_pair not in pairs:
    pairs.append(reference_pair)
reference = ra.pair_label(*reference_pair)

finals = ra.fit_final_models(pairs, splits)
table5 = ra.test_table(finals, splits)
ra.save_table(table5, "table5_test_final_models")
display(table5)

INFO    Refitting random_oversampling · xgboost on the full training split
INFO    Refitting xgb_scale_pos_weight · xgboost on the full training split
INFO    Refitting xgb_focal_loss · xgboost on the full training split
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Refitting smote_enn · xgboost on the full training split
INFO    Refitting smote · xgboost on the full training split
INFO    Refitting none · random_forest on the full training split


,pair,type,auc_pr,auc_pr_lo,auc_pr_hi,delta_vs_selected,delta_lo,delta_hi,significant,threshold,precision,recall,f1,mcc,precision_at_100,tp,fp,fn,tn,total_cost
0,random_oversampling · xgboost,data-level,0.8586,0.7871,0.9243,0.0000,0.0000,0.0000,False,0.8532,0.9077,0.8310,0.8676,0.8683,0.6000,59,6,12,41273,1230.0000
1,xgb_scale_pos_weight · xgboost,algorithm-level,0.8688,0.7982,0.9316,0.0102,-0.0069,0.0302,False,0.7726,0.8923,0.8169,0.8529,0.8535,0.6100,58,7,13,41272,1335.0000
2,xgb_focal_loss · xgboost,algorithm-level,0.8631,0.7913,0.9282,0.0045,-0.0199,0.0349,False,0.3917,0.8615,0.7887,0.8235,0.8240,0.6200,56,9,15,41270,1545.0000
3,smote_enn · xgboost,data-level,0.8328,0.7562,0.9036,-0.0258,-0.0481,-0.0086,True,0.9583,0.8286,0.8169,0.8227,0.8224,0.6000,58,12,13,41267,1360.0000
4,smote · xgboost,data-level,0.8431,0.7679,0.9125,-0.0155,-0.0314,-0.0021,True,0.9659,0.9032,0.7887,0.8421,0.8438,0.6000,56,6,15,41273,1530.0000
5,none · random_forest,untreated,0.8568,0.7840,0.9266,-0.0018,-0.0351,0.0382,False,0.4400,0.8906,0.8028,0.8444,0.8453,0.6300,57,7,14,41272,1435.0000


In [11]:
fig = ra.fig_pr_curves(finals, splits.y_test, table5, reference=reference)
ra.save_figure(fig, "figure_pr_curves_test")
fig.show()

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpj0rlx96h.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp94e0d3v3.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp94e0d3v3
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpj0rlx96h/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 4E81
INFO    Reloading tab 4E81 before return.
INFO    Putting tab 4E81 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

In [12]:
fig = ra.fig_confusion_matrices(finals, splits.y_test)
ra.save_figure(fig, "figure_confusion_matrices_test")
fig.show()

INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpu5b0dzc5.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpgqurcxuh.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpgqurcxuh
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpu5b0dzc5/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 8775
INFO    Reloading tab 8775 before return.
INFO    Putting tab 8775 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worke

## Operating points and error analysis (Table 6)

For the CV-selected model (first row of Table 5). Each row is a different business rule for the threshold, chosen on
validation and applied to test: max F1; minimum expected cost (£100 per missed fraud, £5 per false alarm — the
indicative costs from config, sensitivity-tested in Ch.6); at least 90% precision; and a fixed analyst capacity of
100 alerts/day (the dataset covers 48 hours, so a day ≈ 142k transactions). `alerts_per_day` scales the test alerts
to a full day.

The threshold is a deployment parameter, not a property of the model: this table is the menu an operations team
would choose from, which is the argument for exposing it in the API.

In [13]:
selected = finals[0]
table6 = ra.operating_points(selected, splits)
ra.save_table(table6, "table6_operating_points")
print("Model:", selected.label)
display(table6)

Model: random_oversampling · xgboost


,rule,threshold,alerts_per_day,precision,recall,tp,fp,fn,total_cost
0,max F1,0.8532,223.8507,0.9077,0.8310,59,6,12,1230.0000
1,min expected cost,0.1959,251.4016,0.8219,0.8451,60,13,11,1165.0000
2,precision >= 90%,0.8532,223.8507,0.9077,0.8310,59,6,12,1230.0000
3,100 alerts/day,1.0000,79.2087,1.0000,0.3239,23,0,48,4800.0000


False positives vs false negatives across thresholds, on test. Dotted lines are the Table 6 operating points,
all fixed on validation beforehand — the curves are descriptive and no threshold is read off them. Cost is a separate
chart rather than a second y-axis.

In [14]:
sweep = ra.threshold_sweep(splits.y_test, selected.p_test)
fig = ra.fig_errors_by_threshold(sweep, table6)
ra.save_figure(fig, "figure_errors_by_threshold")
fig.show()
fig = ra.fig_cost_by_threshold(sweep, table6)
ra.save_figure(fig, "figure_cost_by_threshold")
fig.show()

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpl_23jige.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpsunt1b0w.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpsunt1b0w
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpl_23jige/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 4AEE
INFO    Reloading tab 4AEE before return.
INFO    Putting tab 4AEE back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp47sm21jl.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpxce37ng5.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpxce37ng5
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp47sm21jl/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 89A5
INFO    Reloading tab 89A5 before return.
INFO    Putting tab 89A5 back (queue size: 0).
INFO    Waiting for all cleanups to finish.
INFO    Exiting Kaleido.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Closing browser.
INFO    Cancelling tasks.
INFO    Exiting Kalei

## Feature attribution (SHAP, scope-limited)

Mean |SHAP| for the selected model, on validation rows (every fraud plus a 2,000-row legitimate sample), so test
stays reserved for performance. **Interpretation is limited:** V1–V28 are anonymised PCA components, so "V14
matters most" cannot be translated into a real transaction attribute. Say so plainly in Ch.6 §6.4 rather than
over-claiming explainability.

In [15]:
importance = ra.shap_importance(selected, splits, manifest["feature_names"])
ra.save_table(importance, "table7_shap_importance")
fig = ra.fig_shap(importance, selected.label)
ra.save_figure(fig, "figure_shap_importance")
fig.show()

INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    TemporaryDirectory.cleanup() worked.
INFO    shutil.rmtree worked.
INFO    Chromium init'ed with kwargs {}
INFO    Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpwlhkv8lp.
INFO    Opening browser.
INFO    Temp directory created: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp984qciec.
INFO    Temporary directory at: /var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmp984qciec
INFO    Conforming 1 to file:///var/folders/rs/2qv4l0g15gqc_4l76nfrpq4m0000gn/T/tmpwlhkv8lp/index.html
INFO    Getting tab from queue (has 1)
INFO    Got 8142
INFO    Reloading tab 8142 before return.
INFO    Putting tab 8142 back (queue size: 0).
INFO    Waiting 

## Notes for writing Chapter 5

- **CV threshold-dependent metrics are optimistic** (threshold chosen on the scored fold). Quote F1/MCC/precision
  from Table 5 (test, validation-chosen thresholds); quote AUC-PR from either.
- **LightGBM (`lgbm_is_unbalance`)**: with library defaults at 1:578 its scores collapse onto a few distinct values
  saturating at 1.0, so AUC-PR is near the no-skill level while ROC-AUC looks respectable. This is a
  property of the defaults under this imbalance, not of class weighting alone, and there is no untreated LightGBM in
  the grid to separate the two. Report it as such rather than as "LightGBM is a bad technique".
- **RUSBoost** uses imblearn's default base learner, a depth-1 stump; Seiffert et al. (2010) used full C4.5 trees.
  Its weak score should be read with that deviation in mind.
- **SMOTE-Tomek / SMOTE-ENN** were re-run after a fix: the first sweep never cleaned either class, so those rows were
  plain SMOTE. Check the `n_train` column in Table 2 differs from SMOTE's before quoting them.
- **Friedman–Nemenyi** blocks are CV folds, not independent datasets: liberal and low-powered (see §5.3 above).